In [36]:
import io
import json
import base64
import traceback
from IPython.display import HTML, display, Audio
from google.colab import output, userdata
from google import genai
from google.genai import types
import speech_recognition as sr
from pydub import AudioSegment
from gtts import gTTS

# --- 1. SETUP GEMINI CLIENT ---
try:
    # Retrieve API key from Colab Secrets
    api_key = userdata.get('AIchatbot')
    client = genai.Client(api_key=api_key)
except Exception as e:
    print("Warning: GEMINI_API_KEY is not configured in Colab Secrets (🔑 tab). please add it to run the chatbot successfully.")
    client = None

# --- 2. JAVASCRIPT MICROPHONE RECORDER ---
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.srcElement.result);
  reader.readAsDataURL(blob);
});
var record = time => new Promise(async resolve => {
  try {
    stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    recorder = new MediaRecorder(stream);
    chunks = [];
    recorder.ondataavailable = e => chunks.push(e.data);
    recorder.start();
    await sleep(time);
    recorder.onstop = async () => {
      blob = new Blob(chunks);
      text = await b2text(blob);
      resolve(text);
    };
    recorder.stop();
  } catch (err) {
    resolve("ERROR: " + err.toString());
  }
});
"""

def record_audio(seconds=5):
    display(HTML(f"<script>{RECORD_JS}</script>"))
    print(f"\n[Bot] Listening for {seconds} seconds... (Speak now)")
    audio_base64 = output.eval_js(f"record({seconds * 1000})")
    if audio_base64.startswith("ERROR:"):
        raise RuntimeError(f"Microphone access failed: {audio_base64}")
    print("[Bot] Processing speech...")
    header, encoded = audio_base64.split(",", 1)
    return base64.b64decode(encoded)

# --- 3. SPEECH TO TEXT ---
def transcribe_audio(audio_bytes):
    audio_segment = AudioSegment.from_file(io.BytesIO(audio_bytes))
    audio_segment.export("recorded_voice.wav", format="wav")

    wav_io = io.BytesIO()
    audio_segment.export(wav_io, format="wav")
    wav_io.seek(0)

    recognizer = sr.Recognizer()
    with sr.AudioFile(wav_io) as source:
        audio_data = recognizer.record(source)
    # Transcribe audio using Google Speech-to-Text (auto-detects spoken languages)
    return recognizer.recognize_google(audio_data)

# --- 4. MULTILINGUAL GEMINI GENERATION ---
def generate_multilingual_response(prompt):
    if not client:
        raise ValueError("Gemini Client is not configured. Please add your GEMINI_API_KEY secret in the 🔑 panel.")

    # System instructions guide the LLM to reply directly in the user's spoken language,
    # and output structured JSON containing the text response and correct ISO 639-1 language code.
    system_instruction = (
        "You are a friendly, concise multilingual voice assistant. "
        "Detect the language of the prompt and respond in the exact same language. "
        "Keep answers conversational and under 2 sentences long. "
        "You must return a JSON object with 'reply' (string) and 'lang' (two-letter ISO code, e.g., 'es', 'fr', 'en', 'de', 'ja')."
    )

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            response_mime_type="application/json"
        )
    )
    try:
        parsed = json.loads(response.text)
        return parsed.get("reply", ""), parsed.get("lang", "en")
    except Exception:
        return response.text, "en"

# --- 5. TEXT TO SPEECH ---
def play_voice_response(text, lang="en"):
    try:
        tts = gTTS(text=text, lang=lang, slow=False)
    except ValueError:
        # Fallback to English if gTTS doesn't support the language code
        tts = gTTS(text=text, lang="en", slow=False)
    tts.save("bot_response.mp3")
    display(Audio("bot_response.mp3", autoplay=True))

# --- 6. VOICE BOT LOOP ---
def run_voice_bot():
    if not client:
        print("[Bot] Error: Gemini Client not loaded. Please set GEMINI_API_KEY in the 🔑 tab and run again.")
        return
    try:
        # Step A: Capture the speech
        raw_audio = record_audio(seconds=5)

        # Step B: Transcribe
        user_text = transcribe_audio(raw_audio)
        print(f"User: {user_text}")

        # Step C: Ask Gemini
        ai_response, lang_code = generate_multilingual_response(user_text)
        print(f"Bot ({lang_code}): {ai_response}")

        # Step D: Play back the response natively
        play_voice_response(ai_response, lang=lang_code)

    except sr.UnknownValueError:
        print("[Bot] Sorry, I couldn't understand what you said. Please try speaking clearly.")
        print("[Debug] Saved attempt as 'recorded_voice.wav'. You can download it on the left panel to verify.")
    except Exception as e:
        print("[Bot] Error occurred:")
        traceback.print_exc()

# Run the voice bot
run_voice_bot()


[Bot] Listening for 5 seconds... (Speak now)
[Bot] Processing speech...
User: what is mean by Machine learning and review the difference between deep learn
Bot (en): Machine learning is a method where computers learn patterns from data to make decisions without being explicitly programmed. Deep learning is a specialized subset of machine learning that uses multi-layered neural networks to process much more complex data like images and speech.
